# Lekcija 30: Projekcija &mdash; središnja ideja

Ovdje počinje 3. dio kolegija: duboko učenje za računalni vid. Prije izrade ijedne neuronske mreže dobro je zastati i razmotriti ideju kojoj se rijetko posvećuje mnogo pozornosti, unatoč njezinoj temeljnoj važnosti. **Projekcija** je jednostavna ideja: preslikati točke iz jednog prostora u drugi, obično jednostavniji, linearnom kombinacijom njihovih koordinata. Projekcija se provlačila kroz velik dio 1. i 2. dijela &mdash; mnoge dosadašnje tehnike zapravo su bile projekcije. Projekcija je i temelj svega što slijedi &mdash; pokazat će se da neuronske mreže nisu ništa drugo nego *naučene*, *komponirane* projekcije.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Dvije projekcije koje ste već izgradili

**PCA (lekcija 6).** Za oblak točaka, svojstveni vektori njihove matrice kovarijance dali su smjerove na koje možemo projicirati. Projekcija na glavni svojstveni vektor, $y = v^\top x$, svodi svaku 2D točku na jedan broj &mdash; koordinatu duž smjera najveće raspršenosti.

In [ ]:
rng = np.random.default_rng(1)
cloud = rng.multivariate_normal([0, 0], [[3, 1.5], [1.5, 1]], 100)

cov = np.cov(cloud.T)
eigvals, eigvecs = np.linalg.eigh(cov)
principal_direction = eigvecs[:, -1]  # eigenvector of the largest eigenvalue

projected = cloud @ principal_direction

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].scatter(cloud[:, 0], cloud[:, 1], s=15, alpha=0.6)
axes[0].plot([0, 3 * principal_direction[0]], [0, 3 * principal_direction[1]], color='red', linewidth=2)
axes[0].set_aspect('equal')
axes[0].set_title('2D cloud + principal direction')
axes[1].scatter(projected, np.zeros_like(projected), s=15, alpha=0.6)
axes[1].set_yticks([])
axes[1].set_title('Projected onto that direction (1D)')
plt.tight_layout()
plt.show()

**Projekcija kamere (lekcije 24, 25).** $P = K[R|t]$ preslikava 3D točku svijeta u 2D piksel. Svaki redak te matrice, prije završnog dijeljenja u homogenim koordinatama, i sam je linearna projekcija: skalarni umnožak koordinata točke i fiksnog smjera, uz pomak.

Oba primjera imaju isti mehanizam: odaberite smjer $w$ (i možda pomak $b$), a zatim izračunajte $y = w^\top x + b$. Razlikuje se *razlog* odabira smjera: PCA bira $w$ kako bi maksimizirao raspršenost projiciranih podataka (nenadzirani, isključivo geometrijski cilj). Retci matrice kamere određeni su njezinom fizičkom geometrijom. Nijedan odabir nije napravljen radi rješavanja klasifikacijskog problema ili problema prepoznavanja &mdash; ali ništa nas ne sprječava da $w$ odaberemo upravo u tu svrhu.

## Množenje matrica: mnogo projekcija odjednom

Sve prethodno računalo je jedan broj $y = w^\top x$ iz jednog smjera $w$. Posložite $m$ različitih smjerova kao retke matrice $W$ (oblika $m \times n$), i $y = Wx$ računa svih $m$ projekcija odjednom &mdash; jedan skalarni umnožak po retku. Upravo to množenje matrica *jest*: skup istodobnih projekcija. To također znači da su neke naizgled različite linearne operacije iz ranijih lekcija zapravo ista operacija, samo uz drukčiji, strukturirani odabir $W$:

- **Konvolucija** (lekcija 10): svaka izlazna vrijednost skalarni je umnožak malog susjedstva i jezgre &mdash; projekcija. Pomicanje jezgre preko cijelog signala množenje je rijetkom matricom čiji su retci pomaknute kopije jezgre (Toeplitzova matrica).
- **Fourierova transformacija** (lekcija 15): svaka izlazna vrijednost skalarni je umnožak *cijelog* signala i jedne sinusoide fiksne frekvencije &mdash; projekcija na tu frekvenciju. Cijela transformacija množenje je fiksnom matricom sinusa i kosinusa.
- **Homografija ili projekcijska matrica kamere** (lekcije 24, 25): svaka izlazna koordinata skalarni je umnožak homogene točke i jednog retka matrice $H$ ili $P$.

Prva dva primjera provjeravamo izravno: ručno izgradimo matricu i provjerimo reproducira li točno `np.convolve`/`np.fft.fft`.

In [ ]:
signal = rng.normal(size=10)
kernel = np.array([0.25, 0.5, 0.25])  # a small smoothing kernel, as in Lesson 10

# a matrix whose rows are shifted copies of the kernel -- a Toeplitz matrix
n, k = len(signal), len(kernel)
W_conv = np.zeros((n - k + 1, n))
for i in range(n - k + 1):
    W_conv[i, i:i + k] = kernel

via_matmul = W_conv @ signal
via_convolve = np.convolve(signal, kernel, mode='valid')
print(f'max diff, convolution as a matrix multiply: {np.abs(via_matmul - via_convolve).max():.2e}')

N = 8
idx = np.arange(N)
W_dft = np.exp(-2j * np.pi * np.outer(idx, idx) / N)  # row k: the frequency-k sinusoid, sampled at every point

x = rng.normal(size=N)
via_matmul = W_dft @ x
via_fft = np.fft.fft(x)
print(f'max diff, DFT as a matrix multiply:          {np.abs(via_matmul - via_fft).max():.2e}')

Potpuno povezani sloj neuronske mreže upravo je taj isti $y = Wx + b$, s retcima matrice $W$ *naučenima* za rješavanje zadatka umjesto određenima ručno izvedenom strukturom (oblikom jezgre, skupom sinusoida, geometrijom kamere).

## Projekcija za klasifikaciju

Pretpostavimo da cilj nije „maksimizirati raspršenost”, nego „razdvojiti dvije klase”. Ista formula $y = w^\top x + b$ i dalje vrijedi &mdash; projicirajte svaku točku na smjer $w$ i klasificirajte prema *predznaku* dobivene vrijednosti. To je, u cijelosti, jedan umjetni neuron bez aktivacijske funkcije: linearna jezgra od koje se gradi svaki sloj neuronske mreže.

In [ ]:
class_a = rng.normal(loc=[-2, -1], scale=0.8, size=(60, 2))
class_b = rng.normal(loc=[2, 1.5], scale=0.8, size=(60, 2))

# a principled hand-picked direction: point from one class's mean toward the other's
w = class_b.mean(axis=0) - class_a.mean(axis=0)
w = w / np.linalg.norm(w)
midpoint = (class_a.mean(axis=0) + class_b.mean(axis=0)) / 2
b = -w @ midpoint  # threshold: the boundary passes through the midpoint between the classes

score_a = class_a @ w + b
score_b = class_b @ w + b
accuracy = (np.sum(score_a < 0) + np.sum(score_b > 0)) / (len(score_a) + len(score_b))
print(f'w = {np.round(w, 3)}, b = {b:.3f}')
print(f'classification accuracy: {accuracy:.1%}')

In [ ]:
def plot_projection_classifier(ax, class_a, class_b, w, b, title):
    ax.scatter(*class_a.T, s=15, label='class A')
    ax.scatter(*class_b.T, s=15, label='class B')

    # decision boundary: the line {x : w.x + b = 0}, drawn through its closest point to the origin
    foot = -b * w
    perp = np.array([-w[1], w[0]])
    p1, p2 = foot + 5 * perp, foot - 5 * perp
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='black', linewidth=1.5, label='decision boundary')
    ax.arrow(*foot, *w, head_width=0.15, color='red', length_includes_head=True, label='w')

    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_projection_classifier(axes[0], class_a, class_b, w, b, f'Decision boundary\naccuracy={accuracy:.0%}')
axes[0].legend(fontsize=7)

axes[1].scatter(score_a, np.zeros_like(score_a), s=15, label='class A')
axes[1].scatter(score_b, np.zeros_like(score_b), s=15, label='class B')
axes[1].axvline(0, color='black', linewidth=1.5, label='threshold')
axes[1].set_yticks([])
axes[1].set_title('The same points, projected to 1D')
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

Dva prikaza potpuno iste operacije: lijevo je $w$ smjer u izvornom 2D prostoru, a granica odluke pravac okomit na njega; desno je svaka točka projicirana na taj smjer, a klasifikacija je samo pragovanje jednog broja na nuli. 2D prikaz intuitivniji je, ali 1D prikaz može se poopćiti &mdash; u 100 dimenzija ne možemo nacrtati „granicu”, ali „projiciraj na jedan broj pa primijeni prag” i dalje radi potpuno jednako.

## Što jedna projekcija ne može učiniti

Jedna linearna projekcija može proizvesti samo *pravocrtnu* granicu odluke (hiperravninu u višim dimenzijama). Neki skupovi podataka nemaju nikakav pravac koji ih razdvaja, bez obzira na odabir $w$ i $b$ &mdash; klasičan je primjer jedna klasa koja okružuje drugu.

In [ ]:
theta_in = rng.uniform(0, 2 * np.pi, 60)
inner = np.stack([0.5 * np.cos(theta_in), 0.5 * np.sin(theta_in)], axis=1) + rng.normal(0, 0.1, (60, 2))
theta_out = rng.uniform(0, 2 * np.pi, 60)
outer = np.stack([2.0 * np.cos(theta_out), 2.0 * np.sin(theta_out)], axis=1) + rng.normal(0, 0.15, (60, 2))

# search many directions and thresholds for the best possible LINEAR separator
best_acc, best_w, best_b = 0, None, None
for _ in range(2000):
    w_try = rng.normal(size=2)
    w_try /= np.linalg.norm(w_try)
    for b_try in np.linspace(-3, 3, 61):
        s_in, s_out = inner @ w_try + b_try, outer @ w_try + b_try
        acc = max((s_in < 0).sum() + (s_out > 0).sum(), (s_in > 0).sum() + (s_out < 0).sum()) / 120
        if acc > best_acc:
            best_acc, best_w, best_b = acc, w_try, b_try

print(f'best achievable accuracy with ANY single linear projection: {best_acc:.1%}')

fig, ax = plt.subplots(figsize=(4.5, 4.5))
plot_projection_classifier(ax, inner, outer, best_w, best_b, f'Best possible linear boundary\naccuracy={best_acc:.0%}')
plt.show()

Nikakva rotacija ni pomak jednog pravca ne mogu razdvojiti prsten od diska koji okružuje &mdash; najbolje što linearna projekcija može postići jest osrednji kompromis. To je prepreka na koju nailazi svaka potpuno linearna metoda i upravo ona motivira sljedeće dvije lekcije: najprije automatsko *učenje* $w$ i $b$ umjesto ručnog odabira (lekcija 31), a zatim, još važnije, *komponiranje više projekcija s nelinearnostima između njih* (lekcija 32), koje uz dovoljan kapacitet mogu razdvajati područja gotovo proizvoljnog oblika.

### Zadaci

1. U ponavljanju PCA-a projicirajte `cloud` na *drugi* (manji) svojstveni vektor umjesto na glavni. Kako se raspršenost dobivene 1D projekcije uspoređuje s projekcijom na glavni smjer i zašto je to u skladu s onim što govori sama svojstvena vrijednost (lekcija 6)?
2. Izmijenite srednje vrijednosti i raspršenost `class_a`/`class_b` tako da se klase više preklapaju. Kada smjer razlike srednjih vrijednosti `w` prestaje davati visoku točnost i možete li za taj teži slučaj ručno pronaći *bolji* `w` od tog smjera?
3. Za skup podataka s prstenom i diskom, umjesto linearne projekcije pokušajte klasificirati pomoću jednostavne nelinearne funkcije točaka: udaljenosti od ishodišta, $r = \sqrt{x_1^2+x_2^2}$, uz prag na nekoj vrijednosti. Kakvu točnost daje ta jedna nelinearna značajka i što to sugerira o *vrsti* projekcije koja bi stvarno riješila ovaj problem?